<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/agent/gemma4_mcp_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Local Gemma 4 Agent with MCP Tools

This notebook wires a local **Gemma 4** model (served by Ollama) to external **Model Context Protocol (MCP)** tool servers using LlamaIndex. Everything runs on your machine — the LLM never sends your queries to a cloud provider.

**What we build:**
- A **blockchain research agent** using the Blockscout MCP server (read-only: balances, transactions, token transfers, contract info)
- A **math/compute agent** using the Wolfram Alpha MCP server (exact calculations, unit conversions, scientific data)
- A **combined agent** that routes to both tool sets automatically

**Stack:**
- `gemma4:12b` via [Ollama](https://ollama.com) — local LLM
- [`llama-index-tools-mcp`](https://pypi.org/project/llama-index-tools-mcp/) — MCP ↔ LlamaIndex bridge
- [Blockscout MCP server](https://github.com/blockscout/mcp-server) — blockchain data
- [Wolfram Alpha MCP server](https://github.com/modelcontextprotocol/servers/tree/main/src/wolframalpha) — computation

> **Privacy note:** The LLM (Gemma 4) runs 100% locally. MCP tool servers make outbound API calls to Blockscout/Wolfram — only the specific query data (wallet address, equation, etc.) leaves your machine, not your full conversation history.

## Prerequisites

1. **Ollama running** with `gemma4:12b` pulled:
   ```bash
   ollama serve
   ollama pull gemma4:12b
   ```

2. **Node.js** installed (for npx-based MCP servers):
   ```bash
   node --version  # should be >= 18
   ```

3. **Wolfram Alpha API key** (free at [developer.wolframalpha.com](https://developer.wolframalpha.com))

In [ ]:
!pip install llama-index-llms-ollama llama-index-tools-mcp llama-index-core

## 1. Set up the LLM

In [ ]:
import os

def get_gemma_llm(model: str = "gemma4:12b", timeout: float = 180.0):
    """Auto-selects local Ollama or Ollama Cloud based on OLLAMA_CLOUD_API_KEY.
    Set OLLAMA_CLOUD_API_KEY (from the Ollama app) to run without a local server.
    """
    cloud_key = os.environ.get("OLLAMA_CLOUD_API_KEY", "")
    base_url = os.environ.get("OLLAMA_BASE_URL", "")
    model = os.environ.get("OLLAMA_MODEL", model)
    if cloud_key:
        from llama_index.llms.openai_like import OpenAILike
        print(f"Ollama CLOUD — model: {model}")
        return OpenAILike(
            model=model,
            api_base=base_url or "https://api.ollama.com/v1",
            api_key=cloud_key,
            is_chat_model=True,
            is_function_calling_model=True,
            context_window=128_000,
            timeout=timeout,
        )
    from llama_index.llms.ollama import Ollama
    print(f"LOCAL Ollama — model: {model}")
    return Ollama(model=model, base_url=base_url or "http://localhost:11434", request_timeout=timeout)

# Adjust model size to fit your hardware:
# gemma4:2b  (~2 GB)  — fastest / cheapest cloud credits
# gemma4:12b (~8 GB)  — best quality/speed balance  ← default
# gemma4:27b (~18 GB) — highest quality
llm = get_gemma_llm()

# Quick sanity check
resp = llm.complete("In one sentence, what is Ethereum?")
print(resp)

## 2. Connect to the Blockscout MCP Server

[Blockscout](https://blockscout.com) provides a free, read-only blockchain explorer MCP server.
It supports Ethereum, Polygon, Base, Arbitrum, Optimism, and 100+ other EVM chains.

The server is launched on-demand via `npx` — no separate install needed.

In [ ]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

# Launch the Blockscout MCP server as a local subprocess via npx.
# The first run downloads the package (~seconds); subsequent runs use cache.
blockscout_client = BasicMCPClient(
    "npx",
    args=["-y", "@blockscout/mcp-server"],
)

blockscout_spec = McpToolSpec(
    client=blockscout_client,
    # Keep only the safe read-only tools
    allowed_tools=[
        "get_address_info",
        "get_transactions_by_address",
        "get_token_transfers_by_address",
        "get_tokens_by_address",
        "get_transaction_info",
        "get_block_info",
        "get_block_number",
        "lookup_token_by_symbol",
    ],
)

blockscout_tools = await blockscout_spec.to_tool_list_async()
print(f"Loaded {len(blockscout_tools)} Blockscout tools:")
for t in blockscout_tools:
    print(f"  • {t.metadata.name}")

## 3. Connect to the Wolfram Alpha MCP Server

Wolfram Alpha handles exact math, unit conversions, scientific constants, and factual data — things a 12B LLM can hallucinate. By routing computation to Wolfram, the agent gives precise answers.

In [ ]:
import os

# Get a free key at https://developer.wolframalpha.com
WOLFRAM_API_KEY = os.environ.get("WOLFRAM_API_KEY", "YOUR_KEY_HERE")

wolfram_client = BasicMCPClient(
    "npx",
    args=[
        "-y",
        "@modelcontextprotocol/server-wolframalpha",
        "--api-key",
        WOLFRAM_API_KEY,
    ],
)

wolfram_spec = McpToolSpec(client=wolfram_client)
wolfram_tools = await wolfram_spec.to_tool_list_async()
print(f"Loaded {len(wolfram_tools)} Wolfram tools:")
for t in wolfram_tools:
    print(f"  • {t.metadata.name}: {t.metadata.description[:60]}")

## 4. Blockchain Research Agent

A read-only agent that can look up wallet balances, transaction history, and token holdings on any EVM chain. Safe by design — it has no write tools.

In [ ]:
from llama_index.core.agent.workflow import ReActAgent

blockchain_agent = ReActAgent(
    tools=blockscout_tools,
    llm=llm,
    system_prompt=(
        "You are a read-only blockchain research assistant. "
        "Use the available tools to look up on-chain data. "
        "Always state which chain you queried. "
        "Never speculate — if data is unavailable, say so."
    ),
    verbose=True,
)

In [ ]:
# Look up a well-known wallet (Vitalik Buterin's public address)
response = await blockchain_agent.run(
    "What is the ETH balance of 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045 on Ethereum?"
)
print(response)

In [ ]:
# What tokens does a wallet hold?
response = await blockchain_agent.run(
    "What ERC-20 tokens does 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045 hold on Ethereum?"
)
print(response)

In [ ]:
# Look up a specific transaction
response = await blockchain_agent.run(
    "What happened in transaction "
    "0x5c504ed432cb51138bcf09aa5e8a410dd4a1e204ef84bfed1be16dfba1b22060 on Ethereum?"
)
print(response)

## 5. Math & Compute Agent

Routes exact computation to Wolfram Alpha rather than relying on the LLM's arithmetic.

In [ ]:
math_agent = ReActAgent(
    tools=wolfram_tools,
    llm=llm,
    system_prompt=(
        "You are a precise computation assistant. "
        "Always use Wolfram Alpha for calculations — never compute in your head. "
        "Show the exact result returned by Wolfram."
    ),
    verbose=True,
)

In [ ]:
# Compound interest calculation
response = await math_agent.run(
    "If I invest $10,000 at 7% annual interest compounded monthly for 20 years, "
    "what is the final balance?"
)
print(response)

In [ ]:
# Unit conversions
response = await math_agent.run(
    "Convert 1.5 ETH to USD if ETH is $3,200. Also convert the result to euros at 0.92 EUR/USD."
)
print(response)

In [ ]:
# Cryptographic/scientific facts
response = await math_agent.run(
    "How many possible SHA-256 hashes are there? Express it in scientific notation."
)
print(response)

## 6. Combined Agent — blockchain + math in one loop

Give the agent both tool sets and let it decide which to call. Useful for questions that need both on-chain data and computation (e.g. "What is the gas cost of this transaction in USD?").

In [ ]:
all_tools = blockscout_tools + wolfram_tools

combined_agent = ReActAgent(
    tools=all_tools,
    llm=llm,
    system_prompt=(
        "You are a blockchain and finance research assistant. "
        "Use Blockscout tools for on-chain data (read-only) and Wolfram tools for exact computation. "
        "Never make up numbers — always use a tool to verify facts and calculations."
    ),
    verbose=True,
)

print(f"Total tools available: {len(all_tools)}")

In [ ]:
# A question that needs both: blockchain data + math
response = await combined_agent.run(
    "Look up the most recent block on Ethereum. "
    "Then calculate how many blocks are produced per day at the current ~12 second block time, "
    "and how many days ago block 1 was (block 1 was mined on 2015-07-30)."
)
print(response)

In [ ]:
# Researching a token
response = await combined_agent.run(
    "Find the USDC token contract on Ethereum. "
    "Then tell me: if someone holds 50,000 USDC and converts it to ETH at $3,200/ETH, "
    "how much ETH do they get?"
)
print(response)

## 7. Using a remote MCP endpoint (alternative to npx)

If you prefer not to use `npx`, you can connect to any MCP server that exposes an HTTP or SSE endpoint.

In [ ]:
from llama_index.tools.mcp import aget_tools_from_mcp_url

# Example: connect to a self-hosted MCP server via SSE
# tools = await aget_tools_from_mcp_url("http://localhost:8000/sse")

# Example: connect via streamable HTTP
# tools = await aget_tools_from_mcp_url("http://localhost:8000/mcp")

# Example: HuggingFace's hosted MCP server (no key needed for public tools)
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

hf_client = BasicMCPClient("https://huggingface.co/mcp")
hf_spec = McpToolSpec(
    client=hf_client,
    allowed_tools=["search_models", "search_datasets", "search_papers"],
)
hf_tools = await hf_spec.to_tool_list_async()
print(f"HuggingFace tools: {[t.metadata.name for t in hf_tools]}")

In [ ]:
# Use HuggingFace tools with the agent
research_agent = ReActAgent(
    tools=hf_tools,
    llm=llm,
    system_prompt="You are an AI research assistant. Use the tools to find models and papers.",
    verbose=True,
)

response = await research_agent.run(
    "Find the top text-generation models fine-tuned from Gemma on HuggingFace."
)
print(response)

## 8. Managing multiple MCP servers with McpServerManager

For production agents that connect to many MCP servers simultaneously, use `McpServerManager`.

In [ ]:
from llama_index.tools.mcp import McpServerManager

manager = McpServerManager()

# Register servers
manager.add_server(
    "blockscout",
    command="npx",
    args=["-y", "@blockscout/mcp-server"],
)
manager.add_server(
    "wolfram",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-wolframalpha", "--api-key", WOLFRAM_API_KEY],
)

# Get all tools from all servers in one call
all_managed_tools = await manager.get_tools()
print(f"Total managed tools: {len(all_managed_tools)}")

managed_agent = ReActAgent(
    tools=all_managed_tools,
    llm=llm,
    system_prompt="You are a helpful research assistant with access to blockchain and math tools.",
)

## Security checklist

| Item | Status |
|---|---|
| LLM runs locally (Gemma 4 via Ollama) | ✅ No prompts sent to cloud |
| Blockscout tools are **read-only** | ✅ No write operations |
| No wallet private keys in the agent | ✅ Never pass keys to tools |
| Wolfram receives only the math query | ✅ No wallet/personal data |
| `allowed_tools` limits the attack surface | ✅ Only exposed what's needed |

> **Never** give the agent tools that can sign transactions, transfer funds, or manage credentials without explicit human confirmation on each action.

## Next steps

- Add the **Google Drive MCP server** for private RAG over local documents
- Add the **Gmail MCP server** for read-only email research (draft, don't send)
- Combine with **Gemma 4 vision** to analyze on-chain data screenshots
- Deploy the agent behind a **Vercel** endpoint for a web interface